# Flickr30k Final Scientific Closeout

Notebook này đóng hồ sơ khoa học sau khi Held-out Test đã được sử dụng đúng
một lần. Notebook **chỉ đọc output đã khóa**, không đọc lại ảnh/caption
Test, không huấn luyện, không chọn lại checkpoint và không thay đổi phương
pháp hoặc ngưỡng.

## Kaggle input bắt buộc — đúng 6 output

1. `F30K_Global_Val_Lock`;
2. `F30K_DenseMap_Closeout`;
3. `F30K_Val_Report`;
4. `F30K_T3S3_HeldOut_Test_OneTime`;
5. `F30K_T3S3_Deployment_Lock`;
6. `F30K_T3S3_CPU_Proxy`.

Runtime: CPU, Accelerator None, Internet Off; dự kiến dưới 2 phút.


## Cell 1 — Kiểm toán chuỗi khóa và đóng hồ sơ


In [1]:
import csv
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
EXPERIMENT_ID = "F30K_Final_Closeout"
WORK_ROOT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT", f"/kaggle/working/{EXPERIMENT_ID}"
))
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def unique_named(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (f"Cần đúng một {name}", matches)
    return matches[0]

def load(name):
    path = unique_named(name)
    return path, json.loads(path.read_text(encoding="utf-8"))

def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

def verify_manifest(root, manifest):
    rows = manifest["artifacts"]
    for row in rows:
        path = root / row["relative_path"]
        assert path.is_file(), path
        assert path.stat().st_size == int(row["size_bytes"]), path
        assert sha256(path) == row["sha256"], path
    return len(rows)


global_path, global_lock = load("F30K_GLOBAL_VALIDATION_LOCK.json")
dense_path, dense = load("F30K_DenseMap_Closeout_RESULT.json")
val_path, val_report = load("F30K_Val_Report_RESULT.json")
test_path, heldout = load("F30K_T3S3_HeldOut_Test_OneTime_RESULT.json")
deploy_path, deployment = load("F30K_T3S3_Deployment_Lock_RESULT.json")
cpu_path, cpu_proxy = load("F30K_T3S3_CPU_Proxy_RESULT.json")

for result_path in [val_path, test_path, deploy_path, cpu_path]:
    manifest_path = result_path.parent / "MANIFEST.json"
    assert manifest_path.is_file(), manifest_path
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    verify_manifest(result_path.parent, manifest)

assert global_lock["status"] == "locked"
assert global_lock["selected_method"]["pair_id"] == "T3S3_LogitKD"
assert global_lock["held_out_test_authorization_count"] == 1
assert dense["decision"] == "STOP_VALIDATION_NO_GAIN"
assert dense["held_out_test_authorized"] is False
assert val_report["status"] == "complete"
assert val_report["held_out_test_read"] is False

assert heldout["status"] == "complete_locked_no_further_test_access"
assert heldout["test_image_count"] == 1000
assert heldout["test_reference_count"] == 5000
assert heldout["held_out_test_authorization_consumed"] is True
assert heldout["held_out_test_authorization_remaining"] == 0
assert heldout["post_test_retraining_allowed"] is False
assert heldout["post_test_reselection_allowed"] is False
assert heldout["post_test_method_or_threshold_change_allowed"] is False
assert heldout["next_phase_authorized"] == "F30K_Final_Report_Only"
assert heldout["checkpoint_sha256"] == global_lock["selected_method"][
    "deployment_checkpoint_sha256"
]

assert deployment["status"] == "locked_for_conditional_mobile_benchmark"
assert deployment["decision"]["scientific_quality_gate"] == "passed"
assert deployment["decision"]["paired_CIDEr_D_noninferiority"] == "passed"
assert deployment["decision"]["unconditional_production_release"] == "not authorized"
assert cpu_proxy["status"] == "complete"
assert "not an Android device benchmark" in cpu_proxy["role"]

metrics = heldout["metrics_x100_rounded_2dp"]
final_result = {
    "schema_version": "f30k_final_scientific_closeout_v1",
    "created_at_utc": now_utc(),
    "status": "scientific_evaluation_complete",
    "experiment_id": EXPERIMENT_ID,
    "dataset": "Flickr30k Karpathy split",
    "data_counts": {
        "train_images": 29000,
        "validation_images": 1014,
        "held_out_test_images": heldout["test_image_count"],
        "held_out_test_references": heldout["test_reference_count"],
    },
    "selected_method": {
        "pair_id": global_lock["selected_method"]["pair_id"],
        "method": global_lock["selected_method"]["method"],
        "seed": heldout["deployment_seed"],
        "checkpoint_sha256": heldout["checkpoint_sha256"],
        "validation_three_seed_delta_CIDEr_D_x100":
            global_lock["selected_method"]["mean_delta_CIDEr_D_x100"],
        "validation_three_seed_ci95_x100":
            global_lock["selected_method"]["combined_paired_bootstrap_ci95_x100"],
    },
    "held_out_test_metrics_x100": metrics,
    "held_out_test_ce_loss": heldout["metrics_raw"]["ce_loss"],
    "deployment": {
        "precision": deployment["selected_precision"],
        "fp32_reference_MiB": deployment["size"]["fp32_reference_MiB"],
        "selected_MiB": deployment["size"]["selected_MiB"],
        "size_reduction_percent": round(
            deployment["size"]["reduction_fraction"] * 100, 2
        ),
        "int8_noninferiority": deployment["noninferiority"][
            "noninferiority_passed"
        ],
        "strict_verbatim_stability": deployment["decision"][
            "strict_verbatim_stability"
        ],
        "release_scope": "conditional mobile benchmark only",
    },
    "cpu_proxy": {
        "role": cpu_proxy["role"],
        "sample_images": cpu_proxy["sample"]["images"],
        "one_thread_end_to_end_mean_ms":
            cpu_proxy["summaries"]["1"]["end_to_end"]["mean_ms"],
        "four_thread_end_to_end_mean_ms":
            cpu_proxy["summaries"]["4"]["end_to_end"]["mean_ms"],
        "android_measurement": False,
    },
    "negative_result_retained": {
        "pair_id": dense["pair_id"],
        "method": dense["method"],
        "delta_CIDEr_D_x100": next(
            row["delta_mean_x100"]
            for row in dense["validation_summary"]
            if row["metric"] == "CIDEr_D"
        ),
        "decision": dense["decision"],
    },
    "scientific_boundaries": {
        "held_out_test_opened_exactly_once": True,
        "held_out_test_authorization_remaining": 0,
        "post_test_training_or_reselection": False,
        "unconditional_mobile_release": False,
        "android_on_device_benchmark_still_required": True,
    },
    "next_phase_authorized": [
        "scientific report and paper tables from locked outputs",
        "Android on-device benchmark without reopening Test",
    ],
}

result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json(result_path, final_result)

report_path = WORK_ROOT / "F30K_FINAL_CLOSEOUT.md"
report_path.write_text(
    f"""# Flickr30k — kết luận cuối

- Phương pháp được chọn: **T3→S3 Direct Logit-KD**, seed 42.
- Validation ba seed: Δ CIDEr-D = **{global_lock['selected_method']['mean_delta_CIDEr_D_x100']:+.2f}**;
  CI 95% = **[{global_lock['selected_method']['combined_paired_bootstrap_ci95_x100'][0]:.2f};
  {global_lock['selected_method']['combined_paired_bootstrap_ci95_x100'][1]:.2f}]**.
- Held-out Test: BLEU-1 **{metrics['BLEU_1']:.2f}**; BLEU-4
  **{metrics['BLEU_4']:.2f}**; METEOR **{metrics['METEOR']:.2f}**;
  ROUGE-L **{metrics['ROUGE_L']:.2f}**; CIDEr-D **{metrics['CIDEr_D']:.2f}**.
- Gói mô hình hỗn hợp: **{deployment['size']['selected_MiB']:.2f} MiB**,
  giảm **{deployment['size']['reduction_fraction'] * 100:.2f}%** so với FP32.
- INT8 đạt kiểm định không thấp hơn trên Validation, nhưng độ ổn định caption
  nguyên văn không đạt cổng 90%; chưa được phép phát hành sản phẩm vô điều kiện.
- CPU proxy không thay thế benchmark trên thiết bị Android thật.
- Held-out Test đã tiêu thụ đúng một quyền; không được mở hoặc dùng lại để thay
  đổi mô hình.
""",
    encoding="utf-8",
)

artifact_paths = [result_path, report_path]
manifest_path = WORK_ROOT / "MANIFEST.json"
write_json(manifest_path, {
    "schema_version": "f30k_final_closeout_manifest_v1",
    "created_at_utc": now_utc(),
    "artifact_count": len(artifact_paths),
    "artifacts": [
        {
            "relative_path": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in artifact_paths
    ],
})
verify_manifest(WORK_ROOT, json.loads(manifest_path.read_text(encoding="utf-8")))

print("Status                 : SCIENTIFIC EVALUATION COMPLETE")
print("Selected               : T3S3 / Direct Logit-KD / seed 42")
print("Held-out CIDEr-D ×100  :", f"{metrics['CIDEr_D']:.2f}")
print("Test authorization left: 0")
print("Mobile package         :", f"{deployment['size']['selected_MiB']:.2f} MiB")
print("Unconditional release  : NOT AUTHORIZED")
print("Next                    : REPORT + ANDROID ON-DEVICE BENCHMARK")
print("Result                  :", result_path)
print("Report                  :", report_path)
print("Manifest                :", manifest_path)


Status                 : SCIENTIFIC EVALUATION COMPLETE
Selected               : T3S3 / Direct Logit-KD / seed 42
Held-out CIDEr-D ×100  : 61.75
Test authorization left: 0
Mobile package         : 27.18 MiB
Unconditional release  : NOT AUTHORIZED
Next                    : REPORT + ANDROID ON-DEVICE BENCHMARK
Result                  : /kaggle/working/F30K_Final_Closeout/F30K_Final_Closeout_RESULT.json
Report                  : /kaggle/working/F30K_Final_Closeout/F30K_FINAL_CLOSEOUT.md
Manifest                : /kaggle/working/F30K_Final_Closeout/MANIFEST.json
